# 34 - Exact local DMD loss scalar, without geometric matching (NO TRAINING)

**User-approved pilot change:** ALL FOUR arms, including the NEW L1 control, will use the same
minutia-centered/oriented192px crops. Same original23 warmstart,2epochs/arm; no unfair comparison
claim that the old random-crop recipe is unchanged. No pilot is run here. Legacy30/31 stay blocked.

This closes a specific gap in33: its perfect development results use full LSA-R geometry. Here
the ONLY scalar is maximum official masked dense cosine between ONE query minutia patch and the
up-to24 ORIGINAL local descriptors in ONE gallery identity's fixed template. No coordinates,
minutia distances, geometry, Hungarian matching or LSA-R enter that scalar.

Query anchor index is fixed by `hash(seed|epoch|relative_path|dmd_anchor_v1) % anchor_count`, NOT
selected using genuine scores. Each path/impression has an independent anchor choice; no claim
that they are the same anatomical minutia. Gallery anchors remain all cached top-quality<=24
from33. Same-DB, different-identity, duplicate-connected exclusions remain unchanged.

Exact planned loss path: sample aligned192px ORIGINAL/DEGRADED patches on CPU; restoration acts
on192px; DMD sees center128px mapped to[-1,1]. During training this is just a differentiable slice,
NOT CUDA grid_sample backward. Reference anchors are frozen, never extracted from predictions.
Original/reference anchoring is privileged TRAIN supervision, not a deployable degraded-image matcher.

Reuse checksummed33 caches; **no download, NBIS run, optimization, extra images, GPU or training**.
After a runtime reset,02 is needed for source-image integrity and ONE live CPU gradient check.
04/33 need not be rerun when their completed artifacts/cache are intact and library versions match.
This notebook pins the exact completed33 run, never silently picks a newer/better result.

Calibrate THIS local scalar at1% empirical conditional FMR using original clean inner calibration
pairs, separately per DB. NEVER reuse33's tiny LSA-R thresholds or29's cosine thresholds.
Keep the predeclared .80/.70 AUC, .25/.10 attempted TAR and90% pair-matchability floor. Gate FMR
is reported, NOT guaranteed<=1%. These already-inspected main-TRAIN groups are DEVELOPMENT data;
the held-out synthetic test, validation recognition and FVC2004 images remain untouched.

Counterfactual auxiliary objective is unchanged: same nonmated identities before/after, positive
increase of sigmoid((score-threshold)/.05), mean vs worstceil(10%) identity risks. Gradient check
tests the actual local negative score through the192-to128 slice; identity counterfactual must be0.
Finite nonzero score gradients do NOT guarantee a nonzero penalty when no risk increase occurs.

Only a pass may mark eligibility for a NEW matched pilot notebook. No legacy gate is overwritten,
no automatic training starts, and this is neither novelty evidence nor a restoration improvement.
Public SD14 weight label/DPF augmentation and upstream academic/noncommercial license ambiguities
from33 remain unresolved; none is silently converted into a complete data-provenance guarantee.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
DEVICE = torch.device('cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
RUN_SEEDS = {'dry_l1_seed20261110': 20261110}
RUN_NAMES = tuple(RUN_SEEDS)
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('Read-only prerequisites on CPU; DMD inference device selected below.')
MATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'
MATCHER_SETTINGS = {'epochs': 6, 'steps_per_epoch': 200, 'identities_per_batch': 12,
    'learning_rate': 0.0002, 'embedding_dim': 128, 'held_groups_per_db': 8,
    'quality_gate': {'clean_auc': 0.80, 'dry_auc': 0.70, 'clean_tar': 0.25, 'dry_tar': 0.10}}

import ast
import io
import sys
import subprocess
import urllib.request
import zipfile
from scipy.optimize import linear_sum_assignment
import scipy
import torch.nn.functional as F

INFERENCE_DEVICE = torch.device('cpu')
SOURCE33_DIR = PROJECT_ROOT / 'diagnostics' / 'pretrained_dmd_feasibility_v1' / '411576c19ecb440b8392'
print('34: cached local scalar and one live input-gradient check on CPU; no training/GPU/NBIS.')


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if split_name == 'train' and (not path.is_file() or file_digest(path) != record['sha256']):
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')

allowed_train_paths = frozenset(train_sources.relative_path)


In [ ]:

pointer = json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
MATCHER_DIR = Path(pointer['directory'])
if (MATCHER_DIR.parent != MATCHER_ROOT
        or file_digest(MATCHER_DIR / 'configuration.json') != pointer['configuration_sha256']
        or file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']):
    raise RuntimeError('Notebook29 published pointer/integrity mismatch.')
configuration = json.loads((MATCHER_DIR / 'configuration.json').read_text(encoding='utf-8'))
completion = json.loads((MATCHER_DIR / 'completion.json').read_text(encoding='utf-8'))
if (configuration['version'] != 'risk_patch_matcher_v1'
        or configuration['settings'] != MATCHER_SETTINGS or configuration['seed'] != SEED
        or completion['status'] != 'risk_matcher_fit_complete'
        or completion['configuration_sha256'] != pointer['configuration_sha256']
        or completion['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or configuration['selection'] != 'last fixed-budget checkpoint; no gate/validation epoch selection'
        or configuration['train_sources_sha256'] != file_digest(M0_DIR / 'train_sources.csv')
        or configuration['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
        or configuration['simulator_sha256'] != file_digest(SIM_DIR / 'fingerprint_degradation.py')
        or configuration['patch_size'] != SETTINGS['patch_size']):
    raise RuntimeError('Expected unchanged completed29 fixed-budget matcher configuration.')
verify_files(MATCHER_DIR, completion['file_sha256'])
for key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
        ('numpy_version', np.__version__), ('pillow_version', Image.__version__)):
    if configuration[key] != actual:
        raise RuntimeError(f'Matcher library version differs: {key}')
matcher_source = (MATCHER_DIR / 'matcher_source.py').read_bytes().decode('utf-8')
if (hashlib.sha256(matcher_source.encode()).hexdigest() != configuration['model_source_sha256']
        or file_digest(MATCHER_DIR / 'inner_partition.csv') != configuration['partition_sha256']
        or file_digest(MATCHER_DIR / 'checkpoint_helpers.py.txt') != configuration['checkpoint_source_sha256']):
    raise RuntimeError('Frozen matcher source/partition hash differs.')
matcher = types.ModuleType('read_only_failed29_matcher')
matcher.__dict__.update(globals())
exec(compile(matcher_source, str(MATCHER_DIR / 'matcher_source.py'), 'exec'), matcher.__dict__)
partition = pd.read_csv(MATCHER_DIR / 'inner_partition.csv', keep_default_na=False)
expected_partition = matcher.inner_partition(train_sources, MATCHER_SETTINGS['held_groups_per_db'])
pd.testing.assert_frame_equal(partition, expected_partition, check_dtype=False)
latest = json.loads((MATCHER_DIR / 'latest.json').read_text(encoding='utf-8'))
if (latest != completion['checkpoint'] or Path(latest['file']).name != latest['file']
        or latest['configuration_sha256'] != pointer['configuration_sha256']
        or latest['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or file_digest(MATCHER_DIR / latest['file']) != latest['sha256']):
    raise RuntimeError('LAST completed29 checkpoint differs.')
state = torch.load(MATCHER_DIR / latest['file'], map_location='cpu', weights_only=True)
if (state['configuration_sha256'] != pointer['configuration_sha256']
        or state['completed_epochs'] != MATCHER_SETTINGS['epochs']):
    raise RuntimeError('Saved matcher payload differs.')
with torch.random.fork_rng(devices=[]):
    model = matcher.PatchMatcher().to(DEVICE)
model.load_state_dict(state['model'], strict=True)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if any(not torch.isfinite(value).all() for value in model.state_dict().values()):
    raise RuntimeError('Nonfinite frozen matcher state.')
saved_state = {key: value.clone() for key, value in model.state_dict().items()}
saved_history = pd.DataFrame(state['history'])
if saved_history.epoch.tolist() != list(range(1, MATCHER_SETTINGS['epochs'] + 1)):
    raise RuntimeError('Matcher has not trained the full predefined budget.')
del state
calibration = pd.read_csv(MATCHER_DIR / 'calibration_scores.csv', float_precision='round_trip')
gate = pd.read_csv(MATCHER_DIR / 'gate_scores.csv', float_precision='round_trip')
saved_gate = json.loads((MATCHER_DIR / 'quality_gate.json').read_text(encoding='utf-8'))
saved_thresholds = json.loads((MATCHER_DIR / 'thresholds.json').read_text(encoding='utf-8'))
if saved_gate != completion['quality_gate']:
    raise RuntimeError('Saved29 gate attestation differs.')
print('Read-only loaded matcher epoch:', latest['completed_epochs'], '| original gate passed:', saved_gate['passed'])


In [ ]:
AUDIT_SOURCE = 'def panel_records(partition, split):\n    if split not in (\'fit\', \'gate\', \'calibration\'):\n        raise ValueError(\'Only explicit main-TRAIN inner partitions are supported.\')\n    selected = partition[partition.matcher_split.eq(split)]\n    if selected.empty:\n        raise RuntimeError(\'Missing inner partition.\')\n    if split == \'fit\':\n        frames = []\n        for (dataset, db), rows in selected.groupby([\'dataset\', \'db\'], sort=True):\n            groups = sorted(rows.connected_group.unique(),\n                key=lambda key: training.stable_seed(f\'{SEED}|audit_fit_panel|{key}\'))\n            if len(groups) < MATCHER_SETTINGS[\'held_groups_per_db\']:\n                raise RuntimeError(\'Insufficient fit groups for matched audit panel.\')\n            frames.append(rows[rows.connected_group.isin(groups[:MATCHER_SETTINGS[\'held_groups_per_db\']])])\n        selected = pd.concat(frames, ignore_index=True)\n    galleries, probes = [], []\n    for key, rows in selected.groupby(\'identity_key\', sort=True):\n        records = rows.sort_values(\'relative_path\').drop_duplicates(\'sha256\').to_dict(\'records\')\n        if len(records) < 3:\n            raise RuntimeError(f\'Audit requires the same three byte-distinct source impressions: {key}\')\n        galleries.append(records[0])\n        probes.extend(records[1:3])\n    return galleries, probes\n\ndef pair_metadata(partition, split):\n    galleries, probes = panel_records(partition, split)\n    rows = []\n    for condition in (\'identity/none\', \'dry_like/mild\', \'dry_like/moderate\', \'dry_like/severe\'):\n        for probe in probes:\n            for gallery in galleries:\n                if (probe[\'dataset\'], probe[\'db\']) != (gallery[\'dataset\'], gallery[\'db\']):\n                    continue\n                label = int(probe[\'identity_key\'] == gallery[\'identity_key\'])\n                if not label and probe[\'connected_group\'] == gallery[\'connected_group\']:\n                    continue\n                rows.append(dict(dataset=probe[\'dataset\'], db=probe[\'db\'], condition=condition,\n                    probe=probe[\'relative_path\'], gallery=gallery[\'relative_path\'], label=label))\n    return pd.DataFrame(rows)\n\ndef verify_score_metadata(frame, partition, split):\n    keys = [\'dataset\', \'db\', \'condition\', \'probe\', \'gallery\', \'label\']\n    if (frame.empty or frame.duplicated(keys).any() or not np.isfinite(frame.score).all()\n            or not frame.score.between(-1.00001, 1.00001).all() or set(frame.label) != {0, 1}):\n        raise RuntimeError(\'Invalid saved inner verification scores.\')\n    expected = pair_metadata(partition, split)\n    pd.testing.assert_frame_equal(frame[keys].sort_values(keys).reset_index(drop=True),\n        expected[keys].sort_values(keys).reset_index(drop=True), check_dtype=False)\n\ndef crop_details(record):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'Crop audit can read main TRAIN images only.\')\n    dpi = pair_config[\'source_dpi\'][f"{record[\'dataset\']}_{record[\'db\']}"]\n    reference = training.normalized_image(str(DATA_ROOT / relative), dpi)\n    foreground = simulator.foreground_mask(reference)\n    ys, xs = np.where(foreground)\n    if not len(ys):\n        raise RuntimeError(f\'No foreground for frozen crop: {relative}\')\n    crop_seed = training.stable_seed(f\'{SEED}|0|{relative}|crop\')\n    rng = np.random.default_rng(crop_seed)\n    anchor = int(rng.integers(len(ys)))\n    size = SETTINGS[\'patch_size\']\n    height, width = reference.shape\n    top = int(np.clip(int(ys[anchor]) - size // 2, 0, max(0, height - size)))\n    left = int(np.clip(int(xs[anchor]) - size // 2, 0, max(0, width - size)))\n    pad = ((0, max(0, size - height)), (0, max(0, size - width)))\n    padded = np.pad(reference, pad, mode=\'reflect\') if any(value for pair in pad for value in pair) else reference\n    patch = np.array(padded[top:top + size, left:left + size], dtype=np.float32, copy=True)\n    actual = matcher.sample_patches([record], 0, [(\'identity\', \'none\')])[0, 0].numpy()\n    if not np.array_equal(patch, actual):\n        raise RuntimeError(\'Reconstructed crop is not byte-identical to frozen29 sampling.\')\n    coverage = (min(size, height - top) * min(size, width - left)) / (height * width)\n    metadata = dict(relative_path=relative, identity_key=record[\'identity_key\'], dataset=record[\'dataset\'],\n        db=record[\'db\'], crop_seed=crop_seed, normalized_height=height, normalized_width=width,\n        top=top, left=left, patch_size=size, padded_bottom=pad[0][1], padded_right=pad[1][1],\n        crop_image_fraction=coverage, crop_foreground_fraction=float(\n            foreground[top:min(top + size, height), left:min(left + size, width)].mean()),\n        crop_sha256=hashlib.sha256(patch.astype(\'<f4\').tobytes()).hexdigest())\n    return metadata, reference, patch\n\ndef describe_scores(values):\n    values = np.asarray(values, dtype=np.float64)\n    if not len(values) or not np.isfinite(values).all():\n        raise RuntimeError(\'Nonempty finite scores required for descriptive summaries.\')\n    return dict(count=len(values), mean=float(values.mean()), std=float(values.std(ddof=1)) if len(values) > 1 else None,\n        q05=float(np.quantile(values, .05)), median=float(np.median(values)), q95=float(np.quantile(values, .95)))\n\ndef auc_against(genuine, impostors):\n    genuine, impostors = np.asarray(genuine), np.asarray(impostors)\n    if not len(genuine) or not len(impostors):\n        raise RuntimeError(\'Both positive and negative scores required.\')\n    return float(np.mean((genuine[:, None] > impostors).astype(float)\n        + .5 * (genuine[:, None] == impostors)))\n\n@torch.inference_mode()\ndef embed_panel(records, condition):\n    features = []\n    for start in range(0, len(records), 12):\n        images = matcher.sample_patches(records[start:start + 12], 0, [condition]).to(DEVICE)\n        value = model(images)\n        if not torch.isfinite(value).all():\n            raise RuntimeError(\'Nonfinite CPU embeddings.\')\n        features.append(value.cpu())\n    return torch.cat(features)\n\ndef summarize_panel(frame):\n    summaries = []\n    for (split, dataset, db, condition), group in frame.groupby(\n            [\'inner_split\', \'dataset\', \'db\', \'condition\'], sort=True):\n        threshold = saved_thresholds[f\'{dataset}_{db}\']\n        positive = group[group.pool.eq(\'different_impression_genuine\')].score.to_numpy()\n        for pool in (\'different_impression_genuine\', \'same_source_control\', \'same_db_impostor\', \'cross_db_impostor\'):\n            values = group[group.pool.eq(pool)].score.to_numpy()\n            summaries.append(dict(inner_split=split, dataset=dataset, db=db, condition=condition, pool=pool,\n                threshold=threshold, acceptance_fraction=float(np.mean(values >= threshold)),\n                auc_against_cross_impression_genuine=auc_against(positive, values)\n                    if pool.endswith(\'impostor\') else None, **describe_scores(values)))\n    return pd.DataFrame(summaries)\n'
exec(compile(AUDIT_SOURCE, '<matcher-failure-audit>', 'exec'))


In [ ]:

verify_score_metadata(calibration, partition, 'calibration')
verify_score_metadata(gate, partition, 'gate')
thresholds, replay_metrics, replay_summary = matcher.gate_analysis(calibration, gate)
if saved_thresholds != {f'{dataset}_{db}': value for (dataset, db), value in thresholds.items()}:
    raise RuntimeError('Exact CSV replay changed original calibration thresholds.')
if replay_summary != saved_gate:
    raise RuntimeError('Saved gate summary differs from exact saved-score replay.')
prior_metrics = pd.read_csv(MATCHER_DIR / 'gate_metrics.csv', float_precision='round_trip')
pd.testing.assert_frame_equal(replay_metrics, prior_metrics, check_dtype=False, check_exact=True)
print('Exact saved-score replay verified; gate UNCHANGED:')
print(json.dumps(replay_summary, indent=2))
print('Saved fixed-budget learning/runtime/memory history:')
print(saved_history.to_string(index=False))


In [ ]:

EXPECTED_DMD_FILES = {'models/model_zoo.py': '39f24735c03bcbbc0d51f6af75146bd3beb55a17bc25cf1a3b1c8f99bdd54b29', 'models/resnet.py': 'e6d28ed8a9730fc0ce5f1a98d464a129cd976ad91028e4516880908b42688e8d', 'models/units.py': '3c685be67e6ad57c5016cf44f75aa7660ba4675f17277faec470d44b3215a1aa', 'models/inception.py': 'bac60ed9f06c9e2e0869f73bb2bf987bbd3889856e4509f9a0cec5070d05702d', 'models/dataloader_densemnt.py': '70308d0b9a6258f5fb2eb815fa296a987190c0db8d9e5750c00c9df16e9468aa', 'evaluate_mnt.py': '84e7f4e62ebdaf718bc1266367cee5fa7680fd599d585c3cc9398f35a4e5b78f', 'README.md': '2b538fe64ef3510f60512a7b413eb04bbfc66a4c51b4fa00aa67e533eb84ee68', 'LICENCE': '4878c4e24ae4dd9878852724b4ae3313dc9a0599680d6552c1606bbc3d63830c', 'DMD.yaml': '12186bc0a33ab479d9c041aae8e35a180a286b98500c3d990475403732538b3d'}
EXPECTED_DMD_COMMIT = 'feb0ad40e5f340caebb457237d7ef21d5d5724e6'
EXPECTED_WEIGHT_SHA = '6d89dff9e5502cf4534b0fd9e014b3fb4a494a0be904015319c72edd9a7aff48'
EXPECTED_ADAPTER_SHA = '82cde5da161f7c37b58fe1ce069b460965c2a94eb175d12405840fffdc6c0919'
source_config = json.loads((SOURCE33_DIR / 'configuration.json').read_text(encoding='utf-8'))
source33_sha = file_digest(SOURCE33_DIR / 'configuration.json')
source_completion = json.loads((SOURCE33_DIR / 'completion.json').read_text(encoding='utf-8'))
if (source_completion['status'] != 'pretrained_dmd_feasibility_complete'
        or source_completion['configuration_sha256'] != source33_sha
        or SOURCE33_DIR.name != source33_sha[:20]
        or source_config['version'] != 'pretrained_dmd_feasibility_v1'
        or source_config['source_commit'] != EXPECTED_DMD_COMMIT
        or source_config['source_files'] != EXPECTED_DMD_FILES
        or source_config['weight_sha256'] != EXPECTED_WEIGHT_SHA
        or source_config['weight_bytes'] != 179093287
        or source_config['max_anchors'] != 24 or source_config['minimum_anchors'] != 4
        or source_config['adapter_sha256'] != EXPECTED_ADAPTER_SHA
        or source_config['matcher29_configuration_sha256'] != pointer['configuration_sha256']
        or source_config['matcher29_completion_sha256'] != pointer['completion_sha256']
        or source_completion['permits_restoration_training'] is not False):
    raise RuntimeError('Expected the unchanged completed33 source, not a different run/checkpoint.')
verify_files(SOURCE33_DIR,source_completion['file_sha256'])
for name,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),
        ('numpy_version',np.__version__),('pillow_version',Image.__version__),('scipy_version',scipy.__version__)]:
    if source_config[name] != value:
        raise RuntimeError(f'Frozen33 library differs: {name}')
ASSET_ROOT = PROJECT_ROOT / 'pretrained' / 'dmd_sd14_v1' / EXPECTED_DMD_COMMIT
verify_files(ASSET_ROOT,EXPECTED_DMD_FILES)
WEIGHT_PATH = ASSET_ROOT / 'best_model.pth.tar'
if file_digest(WEIGHT_PATH) != EXPECTED_WEIGHT_SHA or WEIGHT_PATH.stat().st_size != 179093287:
    raise RuntimeError('Verified public DMD weights missing/changed; no unsafe fallback.')
adapter33_source = (SOURCE33_DIR / 'adapter.py.txt').read_bytes().decode('utf-8')
if hashlib.sha256(adapter33_source.encode()).hexdigest() != EXPECTED_ADAPTER_SHA:
    raise RuntimeError('Frozen33 adapter changed.')
exec(compile(adapter33_source,'<unchanged33-adapter>','exec'))
local_score,_unused_lsar = load_scoring(ASSET_ROOT / 'evaluate_mnt.py')
del _unused_lsar
dmd_model = load_dmd(ASSET_ROOT,WEIGHT_PATH)
initial_dmd_sha = state_sha(dmd_model)
print('Verified and loaded frozen33:',SOURCE33_DIR.name,'| previous LSA-R gate:',source_completion['quality']['passed'])


In [ ]:
LOCAL_SOURCE = "def query_anchor_index(relative, count, epoch=0):\n    if count < 4:\n        raise RuntimeError('Local surrogate needs at least4 original anchors, same as33.')\n    return training.stable_seed(f'{SEED}|{epoch}|{relative}|dmd_anchor_v1') % count\n\ndef read_cached33(record, family, severity):\n    relative = record['relative_path']\n    if relative not in allowed_train_paths:\n        raise RuntimeError('Cached DMD local analysis is main TRAIN only.')\n    directory = SOURCE33_DIR / 'anchors' / hashlib.sha256(relative.encode()).hexdigest()[:24]\n    expected = dict(relative_path=relative,source_sha256=record['sha256'],\n        configuration_sha256=source33_sha,mindtct_sha256=source_config['nbis_build']['binary_sha256']['mindtct'])\n    if cached_bundle(directory,expected) is None:\n        raise RuntimeError(f'Missing completed33 anchor cache: {relative}. Run33; do not silently re-extract.')\n    with np.load(directory / 'anchors.npz',allow_pickle=False) as archive:\n        anchors = archive['anchors'].copy()\n        shape = archive['image_shape'].copy()\n    parsed = parse_nbis_xyt((directory / 'minutiae.xyt').read_text(encoding='ascii'),*shape)\n    if not np.array_equal(anchors,parsed):\n        raise RuntimeError('Frozen anchors disagree with their XYT source.')\n    if len(anchors) < 4:\n        return dict(status='unmatchable_original_anchor_count',anchors=anchors,feature=None,mask=None)\n    feature_key = hashlib.sha256(f'{relative}|{family}|{severity}'.encode()).hexdigest()[:24]\n    directory = SOURCE33_DIR / 'features' / feature_key\n    expected = dict(relative_path=relative,family=family,severity=severity,\n        configuration_sha256=source33_sha,anchors_sha256=hashlib.sha256(anchors.tobytes()).hexdigest())\n    if cached_bundle(directory,expected) is None:\n        raise RuntimeError(f'Missing completed33 feature cache: {relative}/{family}/{severity}. Run33.')\n    with np.load(directory / 'features.npz',allow_pickle=False) as archive:\n        feature = torch.from_numpy(archive['feature'].copy())\n        mask = torch.from_numpy(archive['mask'].copy())\n    if (feature.shape != (len(anchors),768) or mask.shape != (len(anchors),64)\n            or not torch.isfinite(feature).all() or not torch.isfinite(mask).all()\n            or not ((mask>=0)&(mask<=1)).all()):\n        raise RuntimeError('Invalid cached33 local features.')\n    return dict(status='matched',anchors=anchors,feature=feature,mask=mask)\n\ndef local_identity_score(query_feature,query_mask,gallery_feature,gallery_mask):\n    if (query_feature.shape!=(1,768) or query_mask.shape!=(1,64)\n            or gallery_feature.ndim!=2 or gallery_feature.shape[1]!=768\n            or gallery_mask.shape!=(len(gallery_feature),64) or len(gallery_feature)<4\n            or len(gallery_feature)>24):\n        raise RuntimeError('Expected one local query and one4..24-anchor identity template.')\n    for value in (query_feature,query_mask,gallery_feature,gallery_mask):\n        if not torch.isfinite(value).all():\n            raise RuntimeError('Nonfinite local surrogate input.')\n    matrix = local_score(query_feature[None],gallery_feature[None],query_mask[None],gallery_mask[None],\n        ndim_feat=12,Normalize=False,binary=False)[0]\n    result = matrix.max(dim=1).values.squeeze(0)\n    if not torch.isfinite(result):\n        raise RuntimeError('Nonfinite local identity score.')\n    return result\n\ndef aligned_training_patch(image,anchor):\n    if image.ndim!=4 or image.shape[:2]!=(1,1) or not torch.isfinite(image).all():\n        raise RuntimeError('Expected one finite N1HW source.')\n    point = torch.as_tensor(anchor,device=image.device,dtype=image.dtype).flatten()\n    if len(point)<3 or not torch.isfinite(point).all():\n        raise RuntimeError('Expected a finite fixed original anchor.')\n    height,width = image.shape[-2:]\n    yy,xx = torch.meshgrid(torch.arange(192,device=image.device,dtype=image.dtype)-96,\n        torch.arange(192,device=image.device,dtype=image.dtype)-96,indexing='ij')\n    theta = torch.deg2rad(point[2])\n    sx = point[0]+torch.cos(theta)*xx-torch.sin(theta)*yy\n    sy = point[1]+torch.sin(theta)*xx+torch.cos(theta)*yy\n    grid = torch.stack(((2*sx+1)/width-1,(2*sy+1)/height-1),dim=-1)[None]\n    return F.grid_sample(image-.5,grid,mode='bilinear',padding_mode='zeros',align_corners=False)+.5\n\ndef local_prediction_features(prediction):\n    if (prediction.ndim!=4 or prediction.shape[1:]!=(1,192,192)\n            or not torch.isfinite(prediction).all()):\n        raise RuntimeError('Expected finite N1x192x192 restoration outputs.')\n    output = dmd_model.get_embedding(2*prediction[:,:,32:160,32:160]-1)\n    return output['feature'],output['mask']\n\ndef counterfactual_risks(before,after,threshold,temperature=.05,tail_fraction=.1):\n    if (before.ndim!=1 or before.shape!=after.shape or not len(before)\n            or not torch.isfinite(before).all() or not torch.isfinite(after).all()\n            or not math.isfinite(threshold) or temperature<=0 or not 0<tail_fraction<=1):\n        raise RuntimeError('Finite matched nonmated score vectors and valid risk constants required.')\n    increase = torch.relu(torch.sigmoid((after-threshold)/temperature)\n        -torch.sigmoid((before.detach()-threshold)/temperature))\n    count = max(1,math.ceil(tail_fraction*len(increase)))\n    return increase.mean(),increase.topk(count).values.mean()\n\ndef selected_query(record,bundle):\n    if bundle['status']!='matched':\n        raise RuntimeError('Unmatchable queries must be explicitly counted upstream.')\n    index = query_anchor_index(record['relative_path'],len(bundle['anchors']),0)\n    return index,bundle['feature'][index:index+1],bundle['mask'][index:index+1]\n"
exec(compile(LOCAL_SOURCE,'<dmd-local-scalar>','exec'))


In [ ]:

implementation_sha = 'c324137b71033e2812a9aa37389a0b6852521f07b3af24b643a2d5c68f380c24'
configuration34 = dict(version='dmd_local_surrogate_v1',implementation_sha256=implementation_sha,
    local_source_sha256=hashlib.sha256(LOCAL_SOURCE.encode()).hexdigest(),source33_configuration_sha256=source33_sha,
    source33_completion_sha256=file_digest(SOURCE33_DIR/'completion.json'),
    source29_configuration_sha256=pointer['configuration_sha256'],dmd_weight_sha256=EXPECTED_WEIGHT_SHA,
    query='one hash-selected original anchor/path/epoch; no score selection',
    template='up to24 original gallery anchors; max masked dense cosine; no geometry',
    training_patch=192,dmd_center=128,anchor_epoch=0,all_four_arms_same_aligned_crop=True,
    original23_common_warmstart=True,pilot_epochs_per_arm=2,calibration_fmr=.01,
    temperature=.05,tail_fraction=.1,auxiliary_lambda=.1,triplet_margin=.2,
    torch_version=str(torch.__version__),torch_cuda=str(torch.version.cuda),numpy_version=np.__version__,
    pillow_version=Image.__version__,cpu_threads=torch.get_num_threads(),
    training=False,legacy30_31_authorized=False)
local_sha = hashlib.sha256(json_bytes(configuration34)).hexdigest()
LOCAL_ROOT34 = PROJECT_ROOT/'diagnostics'/'dmd_local_surrogate_v1'
LOCAL_DIR = LOCAL_ROOT34/local_sha[:20]
LOCAL_DIR.mkdir(parents=True,exist_ok=True)
for name,data in {'configuration.json':json_bytes(configuration34),'local_scalar.py.txt':LOCAL_SOURCE.encode()}.items():
    path = LOCAL_DIR/name
    if path.exists() and path.read_bytes()!=data:
        raise RuntimeError(f'Frozen local scalar protocol differs: {path}')
    if not path.exists():
        write_verified(path,data)


In [ ]:

started = time.monotonic()
panels34,frames34,anchor_manifest,control_rows34 = {},{},[],[]
original_cache = {}
for split in ('calibration','gate'):
    galleries,probes = panel_records(partition,split)
    panels34[split]=(galleries,probes)
    for record in galleries+probes:
        relative=record['relative_path']
        bundle=read_cached33(record,'identity','none')
        original_cache[relative]=bundle
        index=query_anchor_index(relative,len(bundle['anchors'])) if bundle['status']=='matched' else None
        anchor_manifest.append(dict(inner_split=split,relative_path=relative,identity_key=record['identity_key'],
            gallery=relative in {row['relative_path'] for row in galleries},status=bundle['status'],
            anchor_count=len(bundle['anchors']),query_anchor_index=index))
        if bundle['status']!='matched':
            print('UNMATCHABLE original-anchor cache:',relative)
    rows=[]
    for family,severity in [('identity','none')]+[('dry_like',level) for level in ('mild','moderate','severe')]:
        for probe in probes:
            relative=probe['relative_path']
            bundle=original_cache[relative] if family=='identity' else read_cached33(probe,family,severity)
            if not np.array_equal(bundle['anchors'],original_cache[relative]['anchors']):
                raise RuntimeError('Condition-specific anchors changed; they must remain original/frozen.')
            if bundle['status']=='matched':
                index,q_feature,q_mask=selected_query(probe,bundle)
                clean=original_cache[relative]
                same=local_score(q_feature[None],clean['feature'][index:index+1][None],
                    q_mask[None],clean['mask'][index:index+1][None],ndim_feat=12,Normalize=False,binary=False)[0,0,0]
                control_rows34.append(dict(inner_split=split,probe=relative,condition=f'{family}/{severity}',
                    same_anchor_dense_score=float(same)))
            for gallery in galleries:
                if (probe['dataset'],probe['db'])!=(gallery['dataset'],gallery['db']):
                    continue
                label=int(probe['identity_key']==gallery['identity_key'])
                if not label and probe['connected_group']==gallery['connected_group']:
                    continue
                target=original_cache[gallery['relative_path']]
                matchable=bundle['status']=='matched' and target['status']=='matched'
                value=float(local_identity_score(q_feature,q_mask,target['feature'],target['mask'])) if matchable else None
                rows.append(dict(dataset=probe['dataset'],db=probe['db'],condition=f'{family}/{severity}',
                    probe=relative,gallery=gallery['relative_path'],label=label,score=value,
                    status='matched' if matchable else 'unmatchable_original_anchor_count'))
        print(f'Cached local scalar {split}/{family}/{severity}: {len(probes)} probes; no geometry/GPU.',flush=True)
    frame=pd.DataFrame(rows)
    keys=['dataset','db','condition','probe','gallery','label']
    pd.testing.assert_frame_equal(frame[keys].sort_values(keys).reset_index(drop=True),
        pair_metadata(partition,split)[keys].sort_values(keys).reset_index(drop=True),check_dtype=False)
    frames34[split]=frame
    write_csv(LOCAL_DIR/f'{split}_scores.csv',frame)
thresholds34,metrics34,summary34=quality_analysis(frames34['calibration'],frames34['gate'])
summary34['scalar']='one fixed query anchor -> max local dense cosine over one gallery identity template; no geometry'
summary34['interpretation']='Development local-loss prerequisite: conditional AUC/FMR, attempted-pair TAR. Not independent evaluation or restoration improvement.'
write_csv(LOCAL_DIR/'query_anchor_manifest.csv',pd.DataFrame(anchor_manifest))
write_csv(LOCAL_DIR/'same_anchor_controls.csv',pd.DataFrame(control_rows34))
write_csv(LOCAL_DIR/'gate_metrics.csv',metrics34)
write_verified(LOCAL_DIR/'thresholds.json',json_bytes(thresholds34))
write_verified(LOCAL_DIR/'quality_gate.json',json_bytes(summary34))
print(metrics34.to_string(index=False))
print(json.dumps(summary34,indent=2))


In [ ]:

chosen=None
for record in panels34['calibration'][1]:
    threshold=thresholds34[f"{record['dataset']}_{record['db']}"]
    if original_cache[record['relative_path']]['status']!='matched' or threshold is None:
        continue
    negatives=[gallery for gallery in panels34['calibration'][0]
        if (record['dataset'],record['db'])==(gallery['dataset'],gallery['db'])
        and record['identity_key']!=gallery['identity_key']
        and record['connected_group']!=gallery['connected_group']
        and original_cache[gallery['relative_path']]['status']=='matched']
    if negatives:
        chosen=(record,negatives,threshold)
        break
if chosen is None:
    gradient34=dict(passed=False,reason='No matchable calibration query with a valid sameDB nonmated template/threshold.')
    print('LOCAL GRADIENT CHECK BLOCKED:',gradient34['reason'])
else:
    record,negative_records,threshold=chosen
    bundle=read_cached33(record,'dry_like','moderate')
    index=query_anchor_index(record['relative_path'],len(bundle['anchors']))
    anchor=bundle['anchors'][index]
    reference=original_train_image(record)
    seed=simulator.case_seed(record['relative_path'],'dry_like','moderate',SEED)
    degraded,_=simulator.degrade(reference,'dry_like','moderate',seed)
    image=torch.from_numpy(degraded.copy())[None,None]
    patch=aligned_training_patch(image,anchor)
    direct=aligned_patches(image,np.array([anchor]))
    crop_difference=float((direct-(2*patch[:,:,32:160,32:160]-1)).abs().max())
    prediction=patch.detach().clone().requires_grad_()
    feature,mask=local_prediction_features(prediction)
    live_scores,cached_scores=[],[]
    for negative in negative_records:
        target=original_cache[negative['relative_path']]
        live_scores.append(local_identity_score(feature,mask,target['feature'],target['mask']))
        cached_scores.append(local_identity_score(bundle['feature'][index:index+1],bundle['mask'][index:index+1],
            target['feature'],target['mask']))
    live_scores=torch.stack(live_scores)
    cached_scores=torch.stack(cached_scores)
    gradient,=torch.autograd.grad(live_scores.mean(),prediction)
    mean_zero,tail_zero=counterfactual_risks(cached_scores,cached_scores,threshold)
    toy_after=(cached_scores.detach()+.01).requires_grad_()
    toy_mean,toy_tail=counterfactual_risks(cached_scores,toy_after,threshold)
    toy_gradient,=torch.autograd.grad(toy_tail,toy_after)
    outside=gradient.detach().clone()
    outside[:,:,32:160,32:160]=0
    gradient34=dict(device='cpu',relative_path=record['relative_path'],query_anchor_index=index,
        negative_identities=len(negative_records),threshold=threshold,
        crop128_max_difference=crop_difference,
        cached33_live_score_max_difference=float((live_scores.detach()-cached_scores).abs().max()),
        finite=bool(torch.isfinite(gradient).all()),absolute_sum=float(gradient.abs().sum()),
        absolute_max=float(gradient.abs().max()),outside_center128_absolute_sum=float(outside.abs().sum()),
        identity_mean_risk=float(mean_zero),identity_tail_risk=float(tail_zero),
        positive_uplift_mean=float(toy_mean.detach()),positive_uplift_tail=float(toy_tail.detach()),
        positive_uplift_tail_gradient_finite=bool(torch.isfinite(toy_gradient).all()),
        positive_uplift_tail_gradient_sum=float(toy_gradient.abs().sum()),
        model_unchanged=state_sha(dmd_model)==initial_dmd_sha,
        parameter_gradients_absent=all(parameter.grad is None for parameter in dmd_model.parameters()),
        note='Actual negative identity score ->192px prediction gradient. Positive-uplift toy checks score-space risk, NOT observed restoration improvement.')
    gradient34['passed']=bool(gradient34['finite'] and gradient34['absolute_sum']>0
        and crop_difference<=2e-5 and gradient34['cached33_live_score_max_difference']<=1e-4
        and gradient34['outside_center128_absolute_sum']==0
        and gradient34['identity_mean_risk']==0 and gradient34['identity_tail_risk']==0
        and gradient34['positive_uplift_tail_gradient_finite']
        and gradient34['positive_uplift_tail_gradient_sum']>0
        and gradient34['model_unchanged'] and gradient34['parameter_gradients_absent'])
    del feature,mask,gradient,prediction,live_scores,cached_scores,toy_after,toy_mean,toy_tail,toy_gradient
write_verified(LOCAL_DIR/'input_gradient.json',json_bytes(gradient34))
print(json.dumps(gradient34,indent=2))


In [ ]:

if state_sha(dmd_model)!=initial_dmd_sha or any(parameter.grad is not None for parameter in dmd_model.parameters()):
    raise RuntimeError('Frozen DMD changed.')
if any(not torch.equal(value,model.state_dict()[key]) for key,value in saved_state.items()):
    raise RuntimeError('Original failed29 model changed.')
verify_files(MATCHER_DIR,completion['file_sha256'])
if (file_digest(MATCHER_DIR/'completion.json')!=pointer['completion_sha256']
        or json.loads((MATCHER_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))!=pointer):
    raise RuntimeError('Original29 completion/pointer changed.')
verify_files(SOURCE33_DIR,source_completion['file_sha256'])
if file_digest(SOURCE33_DIR/'completion.json')!=configuration34['source33_completion_sha256']:
    raise RuntimeError('Completed33 evidence changed.')
verify_files(ASSET_ROOT,EXPECTED_DMD_FILES)
if file_digest(WEIGHT_PATH)!=EXPECTED_WEIGHT_SHA:
    raise RuntimeError('Public DMD weight file changed.')
eligible=bool(summary34['passed'] and gradient34['passed'])
result34=dict(status='dmd_local_surrogate_complete',configuration_sha256=local_sha,
    quality=summary34,gradient_passed=gradient34['passed'],
    permits_new_matched_pilot=eligible,permits_legacy_30_31=False,
    seconds=time.monotonic()-started,
    interpretation='Exact local loss prerequisite on reused development groups; not restoration/novelty evidence.')
names=('configuration.json','local_scalar.py.txt','calibration_scores.csv','gate_scores.csv',
    'query_anchor_manifest.csv','same_anchor_controls.csv','gate_metrics.csv','thresholds.json',
    'quality_gate.json','input_gradient.json')
result34['file_sha256']={name:file_digest(LOCAL_DIR/name) for name in names}
write_verified(LOCAL_DIR/'completion.json',json_bytes(result34))
write_verified(LOCAL_ROOT34/'completed_diagnostic.json',json_bytes(dict(directory=str(LOCAL_DIR),
    configuration_sha256=local_sha,completion_sha256=file_digest(LOCAL_DIR/'completion.json'))))
print('LOCAL DMD SURROGATE CHECK COMPLETE:',LOCAL_DIR)
print('Quality passed:',summary34['passed'],'| gradient passed:',gradient34['passed'])
if eligible:
    print('Eligible for a NEW matched4-arm,2epoch aligned-crop pilot from the common23 warmstart. No training started. Legacy30/31 remain blocked.')
else:
    print('STOP: exact local prerequisite failed. No pilot, gate relaxation, extra epochs or automatic alternate scalar.')
